## Homework 3
#### Jack Neton
#### October 2026

In [160]:
import pandas as pd
import altair as alt

# draw charts with VS Code's built-in Vega-Lite viewer (charts were not loading consistently with the default HTML renderer)
alt.renderers.enable('mimetype');

### Data

Source: City of Seattle, Seattle Pet Licenses (dataset jguv-t9rb), Seattle Open Data Portal, https://data.seattle.gov/Community-and-Culture/Seattle-Pet-Licenses/jguv-t9rb. Accessed October 9, 2026.

Each row is one active pet license issued by the Seattle Animal Shelter. The data has 7 variables:

| Variable | Type | Meaning |
|---|---|---|
| `license_issue_date` | temporal | Date the license was issued |
| `license_number` | nominal (ID) | License number |
| `animal_s_name` | nominal | Pet's name |
| `species` | nominal | Dog, Cat, Goat or Pig |
| `primary_breed` | nominal | Main breed (344 different values) |
| `secondary_breed` | nominal | Second breed, often "Mix"; blank for many pets |
| `zip_code` | nominal (stored as a number) | Owner's ZIP code |

+ Main question: how do licensed dogs and cats differ across Seattle, and where in the city are cats most common?

+ The data has about 38,000 rows

    - Altair will only embed up to 5,000 rows from a data frame
    - We clean the data in pandas and group it into smaller summary tables (counts by day, breed, name and ZIP code)
    - Charts are made from these summary tables using Altair transforms (`filter`, `calculate`, `aggregate`, `window`, `bin`, `timeUnit`)

In [161]:
pets_url = 'https://data.seattle.gov/resource/jguv-t9rb.csv?$limit=50000'
pets = pd.read_csv(pets_url)
pets.shape

(38413, 7)

In [162]:
pets.head()

,license_issue_date,license_number,animal_s_name,species,primary_breed,secondary_breed,zip_code
0,2022-12-12T00:00:00.000,833669,Garbanzo,Cat,Domestic Shorthair,NaN,98115.0
1,2022-12-14T00:00:00.000,S125986,Mia,Cat,Domestic Longhair,Mix,98126.0
2,2023-02-02T00:00:00.000,8042328,Benito,Cat,American Shorthair,Mix,98112.0
3,2023-03-05T00:00:00.000,8029390,Rory,Cat,Domestic Shorthair,Mix,98116.0
4,2023-03-08T00:00:00.000,8026779,FADYA,Cat,Domestic Shorthair,Domestic Shorthair,98107.0


### Step 1: Data Wrangling

+ Check for missing values, rare categories and ZIP codes outside Seattle

In [163]:
pets.isna().sum()   # missing values per column

license_issue_date        0
license_number            0
animal_s_name            20
species                   0
primary_breed             0
secondary_breed       10155
zip_code                122
dtype: int64

In [164]:
pets['species'].value_counts()

species
Dog     25617
Cat     12782
Goat       11
Pig         3
Name: count, dtype: int64

In [165]:
pets['zip_code'].describe()   # ZIP codes should all be 98101-98199 for Seattle

count    38291.000000
mean     98101.015356
std       1182.077885
min       2127.000000
25%      98107.000000
50%      98117.000000
75%      98125.000000
max      99816.000000
Name: zip_code, dtype: float64

*Comments*

+ Missing values

    - About 10,000 pets have no `secondary_breed`, 20 have no name and about 120 have no ZIP code
    - Only missing ZIP codes matter for our questions, and these rows are removed by the ZIP code filter

+ Species

    - Only 11 goats and 3 pigs, too few to compare
    - We keep dogs and cats only

+ ZIP codes

    - `zip_code` is read as a number (e.g. `98115.0`) and ranges from 2127 to 99816
    - About 275 licenses fall outside Seattle's 98101–98199 range, so we remove them
    - Remaining ZIP codes are converted to a 5-digit text field, `ZIP`

+ Dates

    - Almost all licenses were issued between late 2025 and October 2026
    - The table only lists active licenses, so expired licenses are not included
    - A few dozen go back to 2015 (multi-year licenses), which we keep

*Questions*

+ Will the active-license limitation affect the time-based plots? (see Exploratory Plot 2)

In [166]:
pets['license_issue_date'] = pd.to_datetime(pets['license_issue_date'])   # text -> date

clean = pets[pets['species'].isin(['Dog', 'Cat'])]                                   # dogs and cats only
clean = clean[(clean['zip_code'] >= 98101) & (clean['zip_code'] <= 98199)].copy()    # Seattle ZIP codes only
clean['ZIP'] = clean['zip_code'].astype(int).astype(str)                             # 98115.0 -> "98115"
clean.shape

(38002, 8)

In [167]:
# small summary tables for the charts (each well under Altair's 5,000-row limit)
species_counts = pets.groupby('species').size().reset_index(name='Licenses')                        # 4 rows
daily = pets.groupby(['license_issue_date', 'species']).size().reset_index(name='Licenses')         # licenses per day and species
dog_breeds = pets[pets['species'] == 'Dog'].groupby('primary_breed').size().reset_index(name='Count')
names = clean.groupby(['species', 'animal_s_name']).size().reset_index(name='Count')
names = names[names['Count'] >= 10]                                                                  # names used by at least 10 pets
zip_species = clean.groupby(['ZIP', 'species']).size().reset_index(name='Licenses')                 # licenses per ZIP and species

len(daily), len(dog_breeds), len(names), len(zip_species)

(1325, 280, 615, 84)

### Step 2: Exploratory Plots

5 graphics that explore what your data contains and start to describe it and highlight inconsistencies or nuances

#### Exploratory Plot 1

+ Question: how many licenses are there for each species?
+ Bar chart of license counts by species, raw data

In [168]:
bars = alt.Chart(species_counts).mark_bar().encode(
    alt.X('Licenses:Q', axis=alt.Axis(title='Number of Licenses')),
    alt.Y('species:N', sort='-x', axis=alt.Axis(title='Species'))
)

labels = bars.mark_text(align='left', dx=3).encode(
    text='Licenses:Q'
)

bars + labels

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Dogs (about 25,600) outnumber cats (about 12,800) two to one
+ Goats and pigs barely show up

    - Confirms that dropping them is reasonable
    - Count labels added because these bars are too short to read

#### Exploratory Plot 2

+ Question: when were the active licenses issued?
+ Line chart of licenses per month using the `yearmonth` time unit

In [169]:
alt.Chart(daily).mark_line(point=True).encode(
    alt.X('yearmonth(license_issue_date):T', axis=alt.Axis(title='Month Issued')),
    alt.Y('sum(Licenses)', axis=alt.Axis(title='Number of Licenses'))
).properties(width=500)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Near zero from 2015 to late 2024, then a small rise of about 1,000 per month in October–December 2024
+ Jumps to 2,000–3,700 per month starting in October 2025

    - Not an increase in licensing, older licenses have expired out of the data
    - The late-2024 rise is likely two-year licenses that are still valid

+ October 2026 is a partial month
+ Answers the question from Step 1: yes, the timeline only reflects active licenses

    - Time-based plots should be limited to the last 12 months (October 2025 onward)

*Questions*

+ Is there a seasonal pattern within the last 12 months, and is it different for dogs and cats? (see Plot 3)

#### Exploratory Plot 3

+ Question: what are the most common dog breeds?
+ Top 15 dog breeds out of 344, using `window` `rank` and `filter` on the breed counts (same approach as the top directors example in the Week 3 tutorial)

In [170]:
alt.Chart(dog_breeds).mark_bar().transform_window(
    Rank='rank()',
    sort=[alt.SortField('Count', order='descending')]
).transform_filter(
    'datum.Rank <= 15'
).encode(
    alt.X('Count:Q', axis=alt.Axis(title='Number of Dog Licenses')),
    alt.Y('primary_breed:N', sort='-x', axis=alt.Axis(title='Primary Breed'))
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Labrador Retrievers are the most common (about 2,270), followed by short-coat Chihuahuas and Golden Retrievers
+ Some breed values overlap

    - "Mixed Breed, Medium," "Mixed Breed, Large" and "Mix" all describe mixed dogs
    - "Terrier" with no type is very broad
    - Breed counts depend on how owners filled out the form, so we do not use breed in the later plots

#### Exploratory Plot 4

+ Question: how are licenses spread across ZIP codes?
+ Histogram of total licenses per ZIP code, using `aggregate` (sum) by ZIP and bins of 250

In [171]:
alt.Chart(zip_species).mark_bar().transform_aggregate(
    Total='sum(Licenses)',
    groupby=['ZIP']
).encode(
    alt.X('Total:Q', bin=alt.BinParams(step=250), axis=alt.Axis(title='Licenses per ZIP Code (binned)')),
    alt.Y('count()', axis=alt.Axis(title='Number of ZIP Codes'))
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Most Seattle ZIP codes have a few hundred to about 2,000 licenses
+ 98115 and 98103 (north Seattle) have over 3,000
+ The first bin has 21 ZIP codes with fewer than 250 licenses

    - Most of these have only 1–15 licenses (e.g. 98195, the UW campus, and P.O. box ZIP codes)
    - Percentages from counts this small would be unreliable
    - Later comparisons only use ZIP codes with at least 100 licenses

*Questions*

+ Does the mix of dogs and cats differ from one ZIP code to another? (see Plot 2)

#### Exploratory Plot 5

+ Question: on which days and months are licenses issued?
+ Heatmap of month by day of week (two time units), color shows number of licenses, filtered to October 2025 onward based on Exploratory Plot 2

In [172]:
alt.Chart(daily).mark_rect().transform_filter(
    'time(datum.license_issue_date) >= time(datetime(2025, 9, 1))'          # Oct 1, 2025 onward (months are 0-based)
).encode(
    alt.X('month(license_issue_date):O', axis=alt.Axis(title='Month')),
    alt.Y('day(license_issue_date):O', axis=alt.Axis(title='Day of Week')),
    alt.Color('sum(Licenses)', title='Licenses')
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Pattern is weaker than expected

    - Monday is the busiest day (about 5,700 licenses) and Sunday the slowest (about 4,000)
    - Weekend cells are still fairly dark, likely because many licenses are done online

+ October–December are the lightest months
+ Color is harder to compare than position

    - A bar chart of weekday totals would show this more clearly

### Step 3: Brainstorming Preliminary Graphics

3 graphics that investigate potential relationships and insights

+ Plot 1: do dogs and cats get the same names?
+ Plot 2: does the mix of dogs and cats differ by ZIP code? (from Exploratory Plot 4)
+ Plot 3: is there a seasonal pattern, and is it different for dogs and cats? (from Exploratory Plot 2)

#### Plot 1 Discussion

+ Question: do dogs and cats get the same names?
+ Top 10 names for dogs and cats, side by side, using `filter`, `window` `rank` and `filter` on the name counts

In [173]:
dog_names = alt.Chart(names).mark_bar().transform_filter(
    'datum.species == "Dog"'
).transform_window(
    Rank='rank()',
    sort=[alt.SortField('Count', order='descending')]
).transform_filter(
    'datum.Rank <= 10'
).encode(
    alt.X('Count:Q', axis=alt.Axis(title='Number of Dogs')),
    alt.Y('animal_s_name:N', sort='-x', axis=alt.Axis(title='Name'))
).properties(title='Top Dog Names')

cat_names = alt.Chart(names).mark_bar(color='orange').transform_filter(
    'datum.species == "Cat"'
).transform_window(
    Rank='rank()',
    sort=[alt.SortField('Count', order='descending')]
).transform_filter(
    'datum.Rank <= 10'
).encode(
    alt.X('Count:Q', axis=alt.Axis(title='Number of Cats')),
    alt.Y('animal_s_name:N', sort='-x', axis=alt.Axis(title='Name'))
).properties(title='Top Cat Names')

dog_names | cat_names

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Partly. Luna is the most common name for both dogs (241) and cats (108)
+ Lucy, Charlie, Milo and Bella appear on both lists
+ Dog names lean toward human names (Charlie, Daisy, Rosie, Penny)
+ Counts are small, and the two panels use different x-axis scales

    - Easy to compare bar lengths across panels by mistake
    - Interesting to know, but not a strong pattern to pursue

#### Plot 2 Discussion

+ Question: does the mix of dogs and cats differ by ZIP code?
+ Stacked bar chart of licenses per ZIP code, colored by species, sorted by total

In [174]:
alt.Chart(zip_species).mark_bar().encode(
    alt.X('sum(Licenses)', axis=alt.Axis(title='Number of Licenses')),
    alt.Y('ZIP:N', sort='-x', axis=alt.Axis(title='ZIP Code')),
    alt.Color('species:N', title='Species')
).properties(height=600)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ North Seattle ZIP codes (98115, 98103, 98117) have the most pets
+ Dogs outnumber cats in almost every ZIP code
+ Hard to answer the question from this chart

    - Bar length mostly shows how many people live in each ZIP code
    - Cat segments do not share a baseline, so they are hard to compare
    - Small ZIP codes at the bottom add clutter

+ Some downtown ZIP codes (98101, 98104, 98121) seem to have large cat segments for their size

*Questions*

+ Would the share of cats (a percentage) show the ZIP code differences more clearly? (see Improvement 1)

#### Plot 3 Discussion

+ Question: is there a seasonal pattern, and is it different for dogs and cats?
+ Licenses per month by species (`yearmonth` time unit), filtered to October 2025 – September 2026

In [175]:
alt.Chart(daily).mark_line(point=True).transform_filter(
    'datum.species == "Dog" || datum.species == "Cat"'
).transform_filter(
    'time(datum.license_issue_date) >= time(datetime(2025, 9, 1)) && time(datum.license_issue_date) < time(datetime(2026, 9, 1))'   # Oct 2025 - Sep 2026 (months are 0-based)
).encode(
    alt.X('yearmonth(license_issue_date):T', axis=alt.Axis(title='Month Issued')),
    alt.Y('sum(Licenses)', axis=alt.Axis(title='Number of Licenses')),
    alt.Color('species:N', title='Species')
).properties(width=500)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Both lines have the same shape

    - Dogs rise from about 1,100 in October 2025, jump in January, peak around 2,500 in May, then level off near 2,100
    - Cats follow at about half the dog level (550 to 1,250)

+ No difference between dogs and cats, since the lines move together
+ The pattern is more likely renewal timing than a true season

    - The data only has active licenses, so new pets and renewals can't be separated

+ Least useful of the three plots

## Step 4: Refining a Graphic

1 graphic (from above) that is developed across at least 3 iterations, each with brief explanation/justification.

+ We refine Plot 2 because it is the only one of the three that hints at a real pattern (downtown ZIP codes with more cats)
+ Question: which Seattle ZIP codes have the highest share of cats?

#### Improvement 1

+ Changed from counts to percentages, to answer the question from Plot 2

    - `calculate` creates `Cat_Licenses` (count on cat rows, 0 on dog rows)
    - `aggregate` by ZIP sums all licenses (`Total`) and cat licenses (`Cats`)
    - Second `calculate` divides them to get `Pct_Cat`
    - `filter` keeps ZIP codes with at least 100 licenses (from Exploratory Plot 4)
    - Bars sorted by percent

In [176]:
alt.Chart(zip_species).mark_bar().transform_calculate(
    Cat_Licenses='datum.species == "Cat" ? datum.Licenses : 0'
).transform_aggregate(
    Total='sum(Licenses)',
    Cats='sum(Cat_Licenses)',
    groupby=['ZIP']
).transform_calculate(
    Pct_Cat='datum.Cats / datum.Total * 100'
).transform_filter(
    'datum.Total >= 100'
).encode(
    alt.X('Pct_Cat:Q', axis=alt.Axis(title='Percent of Licensed Pets That Are Cats')),
    alt.Y('ZIP:N', sort='-x', axis=alt.Axis(title='ZIP Code'))
).properties(height=500)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Yes, percentages show the difference much more clearly than counts
+ All bars now share a baseline, so cat share can be compared by length
+ Highest cat share is downtown or close to it

    - 98104 (Pioneer Square/International District) about 48%, 98121 (Belltown) 46%, 98101 (Downtown) 45%, 98102 (Eastlake) 44%, 98109 (South Lake Union) 43%

+ Lowest cat share is at the edges of the city

    - 98177 (Broadview) 22%, 98146 (White Center) 23%, 98136 (Fauntleroy) 25%

*Questions*

+ Is cat share related to how big a ZIP code is, rather than where it is? (see Improvement 2)

*Still to improve*

+ No reference point for a typical value (see Improvement 3)
+ With all bars starting at zero, differences between ZIP codes look smaller than they are

#### Improvement 2

+ Question: is cat share related to ZIP code size?
+ Experimental: scatter plot instead of bars

    - Total licenses on the x-axis, percent cats on the y-axis
    - Point size shows number of cats

In [177]:
alt.Chart(zip_species).mark_circle(opacity=0.7).transform_calculate(
    Cat_Licenses='datum.species == "Cat" ? datum.Licenses : 0'
).transform_aggregate(
    Total='sum(Licenses)',
    Cats='sum(Cat_Licenses)',
    groupby=['ZIP']
).transform_calculate(
    Pct_Cat='datum.Cats / datum.Total * 100'
).transform_filter(
    'datum.Total >= 100'
).encode(
    alt.X('Total:Q', axis=alt.Axis(title='Total Licenses in ZIP Code')),
    alt.Y('Pct_Cat:Q', scale=alt.Scale(zero=False), axis=alt.Axis(title='Percent Cats')),
    alt.Size('Cats:Q', title='Number of Cats'),
    tooltip=['ZIP:N', 'Total:Q', 'Pct_Cat:Q']
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ No clear relationship

    - Highest cat-share ZIP codes are mid-sized (500–1,600 licenses)
    - Large north Seattle ZIP codes (98115, 98103) are in the middle at about 30–33%
    - Lowest cat-share ZIP codes (98177, 98146) are also small

+ ZIP code size does not explain cat share, location does
+ Points are unlabeled, so location is lost
+ Size and y-position both encode cats, which is redundant
+ We go back to sorted bars for the next iteration

#### Improvement 3

+ Adds the reference point missing from Improvement 1
+ Bars now diverge from the citywide average instead of starting at zero

    - `calculate` finds each ZIP code's difference from the citywide cat share (about 33%, computed in pandas in the next cell)
    - Second `calculate` labels each ZIP as more cats or more dogs than average
    - Color shows which side each ZIP falls on

In [178]:
city_pct = (clean['species'] == 'Cat').mean() * 100   # citywide % of licensed dogs and cats that are cats
city_pct

np.float64(33.25614441345192)

In [179]:
alt.Chart(zip_species).mark_bar().transform_calculate(
    Cat_Licenses='datum.species == "Cat" ? datum.Licenses : 0'
).transform_aggregate(
    Total='sum(Licenses)',
    Cats='sum(Cat_Licenses)',
    groupby=['ZIP']
).transform_calculate(
    Pct_Cat='datum.Cats / datum.Total * 100'
).transform_filter(
    'datum.Total >= 100'
).transform_calculate(
    Diff=f'datum.Pct_Cat - {city_pct}',
    Leans='datum.Diff > 0 ? "More cats than average" : "More dogs than average"'
).encode(
    alt.X('Diff:Q', axis=alt.Axis(title='Percentage Points Above / Below Citywide Cat Share')),
    alt.Y('ZIP:N', sort='-x', axis=alt.Axis(title='ZIP Code')),
    alt.Color('Leans:N', title=None)
).properties(height=500)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Comments*

+ Zero line now represents the city average
+ Easy to see which ZIP codes lean toward cats or dogs and by how much

    - Downtown ZIP codes are about 10 to 14 points above average
    - West Seattle and the north edge of the city are 6 to 11 points below

*Still to improve*

+ Titles, axis labels and chart size
+ Colors (orange for cats and blue for dogs would be easier to follow)
+ ZIP codes alone don't tell most readers where these areas are, neighborhood names would help

### Step 5: Final Plot and Description

Final, polished plot with one-paragraph discussion of the plot's effectiveness of encodings and design.

+ Fixes the items listed under Improvement 3: title and labels, orange for cats and blue for dogs, and neighborhood names added to each ZIP code

In [180]:
neighborhoods = {
    '98101': 'Downtown', '98102': 'Eastlake / N. Capitol Hill', '98103': 'Fremont / Wallingford', '98104': 'Pioneer Square / ID',
    '98105': 'University District', '98106': 'Delridge / Highland Park', '98107': 'Ballard', '98108': 'Georgetown / Beacon Hill',
    '98109': 'South Lake Union / Queen Anne', '98112': 'Capitol Hill / Madison Park', '98115': 'Ravenna / Wedgwood',
    '98116': 'Alki / Admiral', '98117': 'Crown Hill / NW Ballard', '98118': 'Columbia City / Seward Park',
    '98119': 'Queen Anne', '98121': 'Belltown', '98122': 'Central District', '98125': 'Lake City / Northgate',
    '98126': 'West Seattle / High Point', '98133': 'Bitter Lake / Haller Lake', '98136': 'Fauntleroy / Morgan Junction',
    '98144': 'Mount Baker / Leschi', '98146': 'White Center area', '98177': 'Broadview / North Beach',
    '98178': 'Rainier Beach / Skyway', '98199': 'Magnolia'
}

zip_species['Area'] = zip_species['ZIP'].map(neighborhoods) + ' (' + zip_species['ZIP'] + ')'   # e.g. "Belltown (98121)"
zip_species.head()

,ZIP,species,Licenses,Area
0,98101,Cat,262,Downtown (98101)
1,98101,Dog,317,Downtown (98101)
2,98102,Cat,562,Eastlake / N. Capitol Hill (98102)
3,98102,Dog,709,Eastlake / N. Capitol Hill (98102)
4,98103,Cat,1050,Fremont / Wallingford (98103)


In [181]:
final_bars = alt.Chart(zip_species).transform_calculate(
    Cat_Licenses='datum.species == "Cat" ? datum.Licenses : 0'
).transform_aggregate(
    Total='sum(Licenses)',
    Cats='sum(Cat_Licenses)',
    groupby=['ZIP', 'Area']
).transform_calculate(
    Pct_Cat='datum.Cats / datum.Total * 100'
).transform_filter(
    'datum.Total >= 100'
).transform_calculate(
    Diff=f'datum.Pct_Cat - {city_pct}',
    Leans='datum.Diff > 0 ? "More cats than the city average" : "More dogs than the city average"',
    Label='format(datum.Pct_Cat, ".0f") + "% cats"'
)

bars = final_bars.mark_bar().encode(
    alt.X('Diff:Q', scale=alt.Scale(domain=[-16, 18]),
          axis=alt.Axis(title=f'Percentage points above or below the citywide cat share ({city_pct:.0f}%)')),
    alt.Y('Area:N', sort=alt.EncodingSortField(field='Diff', op='max', order='descending'),
          axis=alt.Axis(title=None, labelLimit=250)),
    alt.Color('Leans:N', title=None,
              scale=alt.Scale(domain=['More cats than the city average', 'More dogs than the city average'],
                              range=['#e08214', '#4a7fb5']),
              legend=alt.Legend(orient='bottom-right')),
    tooltip=['Area:N', 'Total:Q', 'Cats:Q', alt.Tooltip('Pct_Cat:Q', format='.1f', title='% cats')]
)

text = final_bars.mark_text(align='left', dx=4, fontSize=10, color='#444444').transform_filter(
    'datum.Diff > 0'
).encode(
    alt.X('Diff:Q'),
    alt.Y('Area:N', sort=alt.EncodingSortField(field='Diff', op='max', order='descending')),
    text='Label:N'
)

text_left = final_bars.mark_text(align='right', dx=-4, fontSize=10, color='#444444').transform_filter(
    'datum.Diff <= 0'
).encode(
    alt.X('Diff:Q'),
    alt.Y('Area:N', sort=alt.EncodingSortField(field='Diff', op='max', order='descending')),
    text='Label:N'
)

(bars + text + text_left).properties(
    width=500,
    height=560,
    title=alt.TitleParams(
        text="Downtown Seattle ZIP codes have the highest share of cats",
        subtitle=[f'Share of licensed pets that are cats, by ZIP code, compared with the citywide {city_pct:.0f}%.',
                  'ZIP codes with at least 100 active dog or cat licenses. Source: Seattle Pet Licenses, data.seattle.gov (2026).'],
        anchor='start'
    )
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


*Description*

+ Answers the main question: cats are most common downtown and least common at the edges of the city
+ Dense ZIP codes in and around downtown are 43–48% cats, 10–14 points above the citywide 33%

    - Pioneer Square/International District, Belltown, Downtown, Eastlake, South Lake Union

+ Lower-density neighborhoods at the edges of the city are 22–26% cats

    - Broadview, White Center, Fauntleroy, West Seattle

+ The data can't tell us why, but the pattern is consistent with apartments and smaller homes being better suited to cats

*Design choices*

+ Difference from the city average is shown with position along a common scale, the most accurate channel
+ Bars start at the city average instead of zero, so the center line has meaning and small differences are easier to see
+ Bars are sorted, so the ranking reads top to bottom
+ Color repeats the direction of each bar (orange for more cats, blue for more dogs) instead of adding another variable

    - Text labels give the actual percentage, so the chart still works in grayscale

+ ZIP codes replaced with neighborhood names so readers can place each bar
+ Title states the main finding, subtitle explains the comparison, the 100-license minimum and the source

*Transformations used*

+ `filter` (species, minimum count, time window)
+ `calculate` (cat counts, percent, difference from average, labels)
+ `aggregate` (sums by ZIP code)
+ `window` (`rank` for top-k lists)
+ `bin` (histogram)
+ `timeUnit` (`yearmonth`, `month`, `day`)

*Limitations*

+ Only covers licensed pets

    - Cats may be licensed less often than dogs since they are less likely to be seen outside
    - True cat share could be higher in every area

+ Neighborhood names are approximate labels for ZIP codes